In [ ]:
# Import necessary libraries
import sys
import os
sys.path.append('../src')  # Add src directory to path

from pipeline.prompts import get_entity_subparts_by_type
from openai import OpenAI
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import json


In [ ]:
# Set up OpenAI client
# Make sure to set your OpenAI API key as an environment variable
# export OPENAI_API_KEY="your-api-key-here"
client = OpenAI()

# Verify client is working
print("OpenAI client initialized successfully")


In [ ]:
# Load an image
# You can change this path to any image you want to test
image_path = "path/to/your/image.jpg"  # Update this path

# For testing, let's create a sample image if the path doesn't exist
if not os.path.exists(image_path):
    print(f"Image path {image_path} not found. Creating a sample image for testing...")
    # Create a simple test image with some objects
    sample_image = np.ones((400, 600, 3), dtype=np.uint8) * 255  # White background
    # Add some colored rectangles to simulate objects
    sample_image[100:200, 100:200] = [255, 0, 0]  # Red square
    sample_image[200:300, 300:500] = [0, 255, 0]  # Green rectangle
    sample_image[50:150, 400:500] = [0, 0, 255]   # Blue square
    image = sample_image
    print("Using generated sample image")
else:
    # Load the actual image
    image_pil = Image.open(image_path)
    image = np.array(image_pil)
    print(f"Loaded image from {image_path}")

# Display the image
plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title("Input Image")
plt.axis('off')
plt.show()

print(f"Image shape: {image.shape}")
print(f"Image dtype: {image.dtype}")


In [ ]:
# Test get_entity_subparts_by_type function for all artifact types
artifact_types = ['addition', 'removal', 'distortion']
results = {}

print("Testing get_entity_subparts_by_type function...\n")
print("=" * 60)

for artifact_type in artifact_types:
    print(f"\n🔍 Testing artifact type: {artifact_type.upper()}")
    print("-" * 40)
    
    try:
        result = get_entity_subparts_by_type(client, image, artifact_type)
        results[artifact_type] = result
        
        if result and 'error' not in result:
            print(f"✅ Success! GPT-4 Analysis for {artifact_type}:")
            print(f"   Entity: {result.get('entity', 'N/A')}")
            print(f"   Subparts: {result.get('subparts', 'N/A')}")
            
            # Pretty print the JSON
            print(f"\n📋 Full JSON Response:")
            print(json.dumps(result, indent=2))
        else:
            print(f"❌ Error in {artifact_type} analysis:")
            if 'error' in result:
                print(f"   Error type: {result['error']}")
                if 'raw_response' in result:
                    print(f"   Raw response: {result['raw_response']}")
            else:
                print(f"   Result: {result}")
                
    except Exception as e:
        print(f"❌ Exception occurred for {artifact_type}: {str(e)}")
        results[artifact_type] = {'error': str(e)}
    
    print("-" * 40)


In [ ]:
# Summary and comparison of results
print("\n" + "=" * 80)
print("📊 SUMMARY OF RESULTS")
print("=" * 80)

# Create a comparison table
print(f"{'Artifact Type':<15} {'Entity':<15} {'Subparts':<50}")
print("-" * 80)

for artifact_type, result in results.items():
    if result and 'error' not in result:
        entity = result.get('entity', 'N/A')
        subparts = str(result.get('subparts', 'N/A'))
        if len(subparts) > 45:
            subparts = subparts[:42] + "..."
        print(f"{artifact_type:<15} {entity:<15} {subparts:<50}")
    else:
        print(f"{artifact_type:<15} {'ERROR':<15} {'Failed to process':<50}")

print("\n" + "=" * 80)

# Analysis notes
print("\n💡 ANALYSIS NOTES:")
print("- The GPT-4 model analyzes the image and suggests entity parts suitable for each artifact type")
print("- Addition artifacts work best on peripheral/terminal parts (fingers, ears, mirrors)")
print("- Removal artifacts target protruding elements that can be cleanly removed")
print("- Distortion artifacts focus on continuous regions that can be warped (faces, torsos)")
print("\n- Compare the suggested subparts across different artifact types")
print("- Notice how GPT-4 adapts its suggestions based on the specific artifact type requirements")
